# 14. Gün — End-to-End Validation ve Dashboard Finalizasyonu

Bu çalışmada daha önce geliştirilen forecasting sisteminin uçtan uca
doğru ve tutarlı çalışıp çalışmadığı kontrol edilmektedir.

Amaç yeni bir model geliştirmek veya modelleri yeniden eğitmek değildir.

Bunun yerine;

- gerekli veri ve model dosyalarının mevcut olması,
- veri tarihi ile model metadata'sının uyumlu olması,
- kaydedilmiş modellerin doğru şekilde yüklenebilmesi,
- inference pipeline'ın kayıtlı forecast ile aynı sonucu üretmesi,
- prediction interval yapısının geçerli olması,
- forecast değerlerinin Google Trends'in 0–100 aralığında kalması,
- monitoring fonksiyonlarının hatasız çalışması

kontrol edilecektir.

Ayrıca Streamlit dashboard üzerinde forecast ve gerçek veri gösterimi
netleştirilmiş ve gelecekte gerçekleşen değerlerle forward validation
yapılmasına olanak sağlayan bir bölüm eklenmiştir.

In [1]:
# ---------------------------------------------------------
# DAY 14 - END-TO-END VALIDATION
# 1. Gerekli dosyaların kontrolü
# ---------------------------------------------------------
#
# End-to-end validation'a en basit noktadan başlıyoruz:
#
# "Sistemin çalışması için gereken bütün dosyalar gerçekten var mı?"
#
# Çünkü örneğin:
#
# - model dosyası silinmişse,
# - metadata eksikse,
# - forecast CSV bulunamıyorsa
#
# pipeline'ın geri kalanını test etmenin anlamı yoktur.
# ---------------------------------------------------------

from pathlib import Path

import pandas as pd


# ---------------------------------------------------------
# Proje ana klasörünü bulma
# ---------------------------------------------------------
#
# Notebook şu klasörde:
#
# trend-forecast-project/
# └── notebooks/
#     └── 14_end_to_end_validation.ipynb
#
# Notebook'u notebooks/ klasöründen çalıştırdığımız için:
#
# Path.cwd()
#
# bize notebooks/ klasörünü verir.
#
# .parent ise bir üst klasöre çıkar:
#
# trend-forecast-project/
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print("Project root:")
print(PROJECT_ROOT)

Project root:
/Users/nihalapple/Desktop/trend-forecast-project


In [2]:
# ---------------------------------------------------------
# Sistemin ihtiyaç duyduğu dosyalar
# ---------------------------------------------------------
#
# Dictionary kullanıyoruz.
#
# Dictionary:
# "anahtar : değer"
#
# şeklinde veri tutar.
#
# Burada:
#
# anahtar = dosyanın sistemdeki görevi
# değer   = dosyanın Path nesnesi
# ---------------------------------------------------------

required_files = {
    "updated_data": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "google_trends_ai_3y_updated_2026-08-09.csv"
    ),

    "prophet_model": (
        PROJECT_ROOT
        / "models"
        / "chatgpt_prophet_as_of_2026-08-09.json"
    ),

    "xgb_model": (
        PROJECT_ROOT
        / "models"
        / "chatgpt_xgb_as_of_2026-08-09.json"
    ),

    "model_metadata": (
        PROJECT_ROOT
        / "models"
        / "model_metadata_as_of_2026-08-09.json"
    ),

    "interval_metadata": (
        PROJECT_ROOT
        / "models"
        / "prediction_interval_metadata_as_of_2026-08-09.json"
    ),

    "saved_forecast": (
        PROJECT_ROOT
        / "reports"
        / "final_forecast_as_of_2026-08-09.csv"
    ),

    "saved_forecast_with_intervals": (
        PROJECT_ROOT
        / "reports"
        / "final_forecast_with_intervals_as_of_2026-08-09.csv"
    ),
}


# ---------------------------------------------------------
# Dosya varlık kontrolü
# ---------------------------------------------------------
#
# Path.exists():
#
# Dosya gerçekten varsa:
# True
#
# yoksa:
# False
#
# döndürür.
# ---------------------------------------------------------

file_checks = {}


# .items():
#
# Dictionary içerisindeki hem anahtarı
# hem de değeri birlikte dolaşmamızı sağlar.
#
# Örneğin ilk turda:
#
# file_name = "updated_data"
# file_path = .../google_trends_ai_3y_updated_2026-08-09.csv
#
for file_name, file_path in required_files.items():

    file_checks[file_name] = file_path.exists()


# Sonucu daha okunabilir görmek için
# pandas Series'e dönüştürüyoruz.
file_check_series = pd.Series(
    file_checks,
    name="Exists",
)

file_check_series

updated_data                     True
prophet_model                    True
xgb_model                        True
model_metadata                   True
interval_metadata                True
saved_forecast                   True
saved_forecast_with_intervals    True
Name: Exists, dtype: bool

Burada aslında ne test ediyoruz?

Henüz:

"Model doğru tahmin yapıyor mu?"

test etmiyoruz.

Sadece:

Sistemin dependency/artifact'ları hazır mı?

diye bakıyoruz.

Yani bu:

FILE EXISTENCE VALIDATION

In [4]:
# ---------------------------------------------------------
# Tüm gerekli dosyalar var mı?
# ---------------------------------------------------------
#
# file_checks şu tarz bir dictionary:
#
# {
#     "updated_data": True,
#     "prophet_model": True,
#     "xgb_model": True,
#     ...
# }
#
# .values():
# Dictionary'nin yalnızca değerlerini verir:
#
# True, True, True, ...
#
# all():
# İçerisindeki BÜTÜN değerler True ise True döndürür.
#
# Tek bir False bile varsa sonuç False olur.
#
# Örneğin:
#
# all([True, True, True])  -> True
# all([True, False, True]) -> False
# ---------------------------------------------------------

all_required_files_exist = all(
    file_checks.values()
)


if all_required_files_exist:

    print("✅ SYSTEM FILE CHECK: PASS")

else:

    print("❌ SYSTEM FILE CHECK: FAIL")

✅ SYSTEM FILE CHECK: PASS


## 2. Veri ve Model Cutoff Tarihi Kontrolü

Model artifact'larının hangi veri kesimine göre oluşturulduğunu bilmek,
forecast sisteminin tekrar üretilebilirliği açısından önemlidir.

Kaydedilmiş model metadata'sındaki `data_cutoff_date` ile güncel işleme
alınan veri setinin son tarihi karşılaştırılacaktır.

Bu iki tarihin farklı olması, model ile veri versiyonlarının birbirine
karışmış olabileceğini gösterebilir.

In [5]:
# ---------------------------------------------------------
# 2. DATA CUTOFF VALIDATION
# ---------------------------------------------------------

import json


# ---------------------------------------------------------
# Güncel veri setini yükleme
# ---------------------------------------------------------
#
# parse_dates=["date"]:
# date kolonunu string yerine pandas datetime yapar.
#
# index_col="date":
# Tarihi DataFrame'in index'i olarak kullanır.
# ---------------------------------------------------------

updated_data = pd.read_csv(
    required_files["updated_data"],
    parse_dates=["date"],
    index_col="date",
)


# ---------------------------------------------------------
# Model metadata dosyasını yükleme
# ---------------------------------------------------------
#
# JSON:
# Python dictionary yapısına benzeyen,
# yapılandırılmış bilgi saklama formatıdır.
#
# json.load():
# Açılmış JSON dosyasını okuyarak
# Python dictionary'ye dönüştürür.
# ---------------------------------------------------------

with open(
    required_files["model_metadata"],
    "r",
    encoding="utf-8",
) as metadata_file:

    model_metadata = json.load(
        metadata_file
    )


# ---------------------------------------------------------
# Verinin son tarihini bulma
# ---------------------------------------------------------
#
# .index.max():
# Date index'indeki en büyük,
# yani en güncel tarihi getirir.
# ---------------------------------------------------------

data_cutoff_date = updated_data.index.max()


# ---------------------------------------------------------
# Metadata'daki cutoff tarihini alma
# ---------------------------------------------------------
#
# Metadata içinde tarih string olarak tutuluyor:
#
# "2026-08-09"
#
# pd.Timestamp():
# Bu string'i pandas tarih nesnesine çevirir.
#
# Böylece iki tarihi güvenli biçimde
# karşılaştırabiliriz.
# ---------------------------------------------------------

metadata_cutoff_date = pd.Timestamp(
    model_metadata["data_cutoff_date"]
)


print(
    "Data cutoff:    ",
    data_cutoff_date.date(),
)

print(
    "Metadata cutoff:",
    metadata_cutoff_date.date(),
)

Data cutoff:     2026-08-09
Metadata cutoff: 2026-08-09


In [6]:
# ---------------------------------------------------------
# İki tarih aynı mı?
# ---------------------------------------------------------

cutoff_matches = (
    data_cutoff_date
    == metadata_cutoff_date
)


if cutoff_matches:

    print(
        "✅ DATA / METADATA CUTOFF CHECK: PASS"
    )

else:

    print(
        "❌ DATA / METADATA CUTOFF CHECK: FAIL"
    )

✅ DATA / METADATA CUTOFF CHECK: PASS


## 3. Kaydedilmiş Modellerin Yüklenme Kontrolü

Final forecasting sisteminde ChatGPT tahmini, daha önce eğitilip `models/`
klasörüne kaydedilmiş Prophet ve XGBoost modellerinin yeniden yüklenmesiyle
üretilmektedir.

Bu aşamada modeller yeniden eğitilmeden:

- Prophet model dosyasının deserialize edilebildiği,
- XGBoost model dosyasının yüklenebildiği

kontrol edilmektedir.

Bu kontrolün başarılı olması, kaydedilmiş model artifact'larının inference
aşamasında kullanılabilir durumda olduğunu gösterir.

In [7]:
# ---------------------------------------------------------
# 3. SAVED MODEL LOADING VALIDATION
# ---------------------------------------------------------
#
# Bu aşamada modeli YENİDEN EĞİTMİYORUZ.
#
# models/ klasörüne daha önce kaydettiğimiz:
#
# - Prophet
# - XGBoost
#
# modellerini dosyadan tekrar yüklemeyi deniyoruz.
#
# Amaç:
#
# "Model artifact'ları gerçekten kullanılabilir mi?"
# ---------------------------------------------------------


# ---------------------------------------------------------
# Prophet modelini JSON'dan yüklemek için
# ---------------------------------------------------------
#
# Prophet modeli normal json.load() ile doğrudan
# modele dönüşmez.
#
# model_from_json():
# Prophet'in kendi serialization mekanizmasıdır.
# Kaydedilmiş JSON metnini tekrar Prophet model
# nesnesine dönüştürür.
# ---------------------------------------------------------

from prophet.serialize import model_from_json


# ---------------------------------------------------------
# XGBoost modelini yüklemek için
# ---------------------------------------------------------

from xgboost import XGBRegressor


# ---------------------------------------------------------
# Prophet modelini yükleme
# ---------------------------------------------------------

with open(
    required_files["prophet_model"],
    "r",
    encoding="utf-8",
) as prophet_file:

    # .read():
    # JSON dosyasının tüm içeriğini string olarak okur.
    #
    # model_from_json():
    # Bu JSON string'ini tekrar eğitilmiş
    # Prophet model nesnesine dönüştürür.
    loaded_prophet_model = model_from_json(
        prophet_file.read()
    )


# ---------------------------------------------------------
# XGBoost modelini yükleme
# ---------------------------------------------------------
#
# Önce boş bir XGBRegressor nesnesi oluşturuyoruz.
#
# Daha sonra load_model():
# JSON dosyasındaki eğitilmiş model yapısını
# bu nesnenin içerisine yükler.
# ---------------------------------------------------------

loaded_xgb_model = XGBRegressor()

loaded_xgb_model.load_model(
    required_files["xgb_model"]
)


# ---------------------------------------------------------
# Yüklenen nesneleri kontrol etme
# ---------------------------------------------------------

print(
    "Prophet model type:",
    type(loaded_prophet_model).__name__,
)

print(
    "XGBoost model type:",
    type(loaded_xgb_model).__name__,
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Prophet model type: Prophet
XGBoost model type: XGBRegressor


In [8]:
# ---------------------------------------------------------
# Model loading sonucu
# ---------------------------------------------------------
#
# isinstance(nesne, sınıf):
#
# Bir Python nesnesinin beklediğimiz sınıftan
# olup olmadığını kontrol eder.
#
# Örneğin:
#
# isinstance(loaded_xgb_model, XGBRegressor)
#
# True ise gerçekten bir XGBRegressor nesnesidir.
# ---------------------------------------------------------

prophet_loaded_correctly = (
    type(loaded_prophet_model).__name__ == "Prophet"
)

xgb_loaded_correctly = isinstance(
    loaded_xgb_model,
    XGBRegressor,
)


# İki model de doğru yüklenmiş olmalı.
models_loaded_correctly = all(
    [
        prophet_loaded_correctly,
        xgb_loaded_correctly,
    ]
)


if models_loaded_correctly:

    print(
        "✅ SAVED MODEL LOADING CHECK: PASS"
    )

else:

    print(
        "❌ SAVED MODEL LOADING CHECK: FAIL"
    )

✅ SAVED MODEL LOADING CHECK: PASS


## 4. Pipeline Reproducibility Kontrolü

Bu aşamada `src/pipeline.py` içerisindeki final inference pipeline'ı
çalıştırılarak kaydedilmiş modellerden yeniden forecast üretilmektedir.

Yeniden üretilen forecast, daha önce `reports/` klasörüne kaydedilmiş
9 Ağustos 2026 cutoff tarihli final forecast ile karşılaştırılacaktır.

Amaç modelleri yeniden eğitmek değil; aynı veri, aynı model artifact'ları
ve aynı metadata kullanıldığında sistemin aynı tahmini tekrar üretip
üretemediğini doğrulamaktır.

Bu özellik sistemin tekrar üretilebilirliği (`reproducibility`) açısından
önemlidir.

In [10]:
# ---------------------------------------------------------
# Proje ana klasörünü Python import yoluna ekleme
# ---------------------------------------------------------
#
# Notebook notebooks/ klasöründe çalışıyor.
#
# src/ klasörü ise bir üst klasörde:
#
# trend-forecast-project/
# ├── notebooks/
# └── src/
#
# Bu nedenle Python'a proje root'unu
# import yapılabilecek klasörler arasına ekliyoruz.
# ---------------------------------------------------------

import sys


# str(PROJECT_ROOT):
# Path nesnesini normal dosya yolu string'ine çevirir.
#
# sys.path:
# Python'ın import yaparken baktığı klasörlerin listesidir.
#
# insert(0, ...):
# Proje root'unu listenin en başına ekler.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# Artık Python şunu görebilir:
#
# trend-forecast-project/src/pipeline.py
from src.pipeline import generate_final_forecast

In [11]:
# ---------------------------------------------------------
# 4. PIPELINE REPRODUCIBILITY VALIDATION
# ---------------------------------------------------------
#
# Şimdi artık tek tek Prophet veya XGBoost'u
# kendimiz çalıştırmıyoruz.
#
# Bunun yerine gerçek uygulamamızda kullandığımız:
#
# src/pipeline.py
#
# içerisindeki generate_final_forecast()
# fonksiyonunu çağırıyoruz.
#
# Böylece dashboard'un da kullandığı gerçek
# inference akışını test etmiş oluyoruz.
# ---------------------------------------------------------

import numpy as np




# ---------------------------------------------------------
# Pipeline ile forecast'u yeniden üretme
# ---------------------------------------------------------
#
# updated_data:
# 9 Ağustos 2026'ya kadar olan güncel veri.
#
# models_dir:
# Kaydedilmiş Prophet ve XGBoost modellerinin
# bulunduğu klasör.
#
# metadata_path:
# Hangi trendin hangi modeli kullandığını
# açıklayan metadata dosyası.
# ---------------------------------------------------------

pipeline_forecast = generate_final_forecast(
    data=updated_data,
    models_dir=PROJECT_ROOT / "models",
    metadata_path=required_files["model_metadata"],
)


# Önce yeniden üretilen forecast'u görelim.
pipeline_forecast

,ChatGPT,Gemini,Claude
Date,,,
2026-08-16,70.248219,40.0,15.0
2026-08-23,70.850414,40.0,15.0
2026-08-30,71.373637,40.0,15.0
2026-09-06,72.022927,40.0,15.0


In [14]:
# ---------------------------------------------------------
# Daha önce kaydedilmiş final forecast'u yükleme
# ---------------------------------------------------------
#
# Bu dosya Day 9'da oluşturduğumuz:
#
# reports/
# final_forecast_as_of_2026-08-09.csv
#
# dosyasıdır.
#
# Yani karşılaştırmanın diğer tarafında
# daha önce dondurduğumuz forecast bulunuyor.
# ---------------------------------------------------------

saved_forecast = pd.read_csv(
    required_files["saved_forecast"],
    parse_dates=["Date"],
    index_col="Date",
)


saved_forecast

,ChatGPT,Gemini,Claude
Date,,,
2026-08-16,70.248219,40.0,15.0
2026-08-23,70.850414,40.0,15.0
2026-08-30,71.373637,40.0,15.0
2026-09-06,72.022927,40.0,15.0


In [16]:
# ---------------------------------------------------------
# Forecast değerlerini karşılaştırma
# ---------------------------------------------------------
#
# np.allclose():
#
# İki sayısal array'in değerlerinin birbirine
# yeterince yakın olup olmadığını kontrol eder.
#
# Neden doğrudan:
#
# pipeline_forecast == saved_forecast
#
# kullanmıyoruz?
#
# Çünkü floating-point sayılarda çok küçük
# hesaplama farkları oluşabilir.
#
# Örneğin:
#
# 70.2482190001
# 70.2482190002
#
# pratikte aynı tahmin olmasına rağmen
# doğrudan == kullanıldığında problem çıkabilir.
#
# np.allclose() bu yüzden sayısal model
# çıktılarında daha güvenli bir karşılaştırmadır.
# ---------------------------------------------------------

forecast_values_match = np.allclose(
    pipeline_forecast,
    saved_forecast,
)


print(
    "Forecast values match:",
    forecast_values_match,
)

Forecast values match: True


In [17]:
# ---------------------------------------------------------
# Index / tarih kontrolü
# ---------------------------------------------------------
#
# .equals():
# İki pandas Index nesnesinin hem değerlerinin
# hem sırasının aynı olup olmadığını kontrol eder.
# ---------------------------------------------------------

forecast_dates_match = (
    pipeline_forecast.index.equals(
        saved_forecast.index
    )
)


# ---------------------------------------------------------
# Kolon kontrolü
# ---------------------------------------------------------
#
# Örneğin her ikisinde de:
#
# ChatGPT
# Gemini
# Claude
#
# aynı sırada bulunmalı.
# ---------------------------------------------------------

forecast_columns_match = (
    pipeline_forecast.columns.equals(
        saved_forecast.columns
    )
)


print(
    "Forecast dates match:",
    forecast_dates_match,
)

print(
    "Forecast columns match:",
    forecast_columns_match,
)

Forecast dates match: True
Forecast columns match: True


In [18]:
# ---------------------------------------------------------
# Genel reproducibility sonucu
# ---------------------------------------------------------
#
# Forecast'un aynı kabul edilmesi için:
#
# 1. Sayısal değerler aynı olmalı
# 2. Forecast tarihleri aynı olmalı
# 3. Kolonlar aynı olmalı
#
# Bu üç koşulun da True olmasını istiyoruz.
# ---------------------------------------------------------

pipeline_reproducible = all(
    [
        forecast_values_match,
        forecast_dates_match,
        forecast_columns_match,
    ]
)


if pipeline_reproducible:

    print(
        "✅ PIPELINE REPRODUCIBILITY CHECK: PASS"
    )

else:

    print(
        "❌ PIPELINE REPRODUCIBILITY CHECK: FAIL"
    )

✅ PIPELINE REPRODUCIBILITY CHECK: PASS


Yani 9 Ağustos’ta oluşturduğumuz model snapshot’ları ve metadata bugün yeniden yüklendiğinde aynı 4 haftalık tahmini üretebiliyor.

## 5. Prediction Interval ve Domain Kontrolü

Bu aşamada final forecast için kullanılan %80 ampirik prediction interval
yapısının geçerli olup olmadığı kontrol edilmektedir.

Her trend için:

- interval metadata'sının okunabilmesi,
- alt sınırın tahminden küçük veya eşit olması,
- üst sınırın tahminden büyük veya eşit olması,
- forecast ve interval değerlerinde eksik (`NaN`) bulunmaması,
- bütün değerlerin Google Trends'in doğal 0–100 aralığında kalması

kontrol edilecektir.

Bu test, dashboard üzerinde gösterilen forecast belirsizlik aralıklarının
yapısal olarak geçerli olduğunu doğrular.

In [22]:
# ---------------------------------------------------------
# 5. PREDICTION INTERVAL + DOMAIN VALIDATION
# ---------------------------------------------------------
#
# Day 12'de prediction interval'ları doğrudan
# Prophet'in kendi interval'ından almamıştık.
#
# Bunun yerine:
#
# Cross-validation residual dağılımı
#           ↓
# Q10 ve Q90
#           ↓
# Merkezi %80 ampirik prediction interval
#
# oluşturmuştuk.
#
# Residual tanımımız:
#
# residual = actual - prediction
#
# Bu nedenle:
#
# Lower = Forecast + Q10
# Upper = Forecast + Q90
#
# kullanıyoruz.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Interval metadata dosyasını yükleme
# ---------------------------------------------------------

with open(
    required_files["interval_metadata"],
    "r",
    encoding="utf-8",
) as interval_file:

    interval_metadata = json.load(
        interval_file
    )

print(
    "Interval level:",
    interval_metadata["interval_level"],
)

print(
    "Method:",
    interval_metadata["method"],
)

Interval level: 0.8
Method: empirical_cv_residual_quantiles


In [24]:
# ---------------------------------------------------------
# Her trend için interval oluşturma ve kontrol etme
# ---------------------------------------------------------
#
# pipeline_forecast kolonları:
#
# ChatGPT
# Gemini
# Claude
#
# Interval metadata anahtarları ise:
#
# chatgpt
# gemini
# claude
#
# Bu yüzden küçük bir eşleştirme dictionary'si
# kullanıyoruz.
# ---------------------------------------------------------

trend_name_map = {
    "ChatGPT": "chatgpt",
    "Gemini": "gemini",
    "Claude": "claude",
}


# Her trendin validation sonucunu burada tutacağız.
interval_validation_results = {}


for forecast_column, metadata_key in trend_name_map.items():

    # -----------------------------------------------------
    # O trendin final point forecast'u
    # -----------------------------------------------------

    forecast = pipeline_forecast[
        forecast_column
    ]


    # -----------------------------------------------------
    # O trende ait residual quantile bilgileri
    # -----------------------------------------------------

    trend_interval_metadata = (
        interval_metadata["trends"][metadata_key]
    )

    q10 = float(
        trend_interval_metadata["q10"]
    )

    q90 = float(
        trend_interval_metadata["q90"]
    )


    # -----------------------------------------------------
    # Prediction interval
    # -----------------------------------------------------
    #
    # residual = actual - prediction olduğu için:
    #
    # Lower = Forecast + Q10
    # Upper = Forecast + Q90
    # -----------------------------------------------------

    lower = (
        forecast + q10
    ).clip(
        lower=0,
        upper=100,
    )

    upper = (
        forecast + q90
    ).clip(
        lower=0,
        upper=100,
    )


    # -----------------------------------------------------
    # 1. Lower <= Forecast kontrolü
    # -----------------------------------------------------
    #
    # (lower <= forecast)
    #
    # her tarih için True/False üretir.
    #
    # .all():
    # bütün tarihler True ise tek bir True döndürür.
    # -----------------------------------------------------

    lower_check = (
        lower <= forecast
    ).all()


    # -----------------------------------------------------
    # 2. Forecast <= Upper kontrolü
    # -----------------------------------------------------

    upper_check = (
        forecast <= upper
    ).all()


    # -----------------------------------------------------
    # 3. NaN kontrolü
    # -----------------------------------------------------
    #
    # pd.concat():
    # Forecast, Lower ve Upper serilerini
    # yan yana tek DataFrame'e getirir.
    #
    # .isna():
    # Eksik değerleri True olarak işaretler.
    #
    # .any().any():
    # DataFrame'in herhangi bir yerinde
    # en az bir NaN var mı diye kontrol eder.
    #
    # Başına "not" koyduğumuz için:
    #
    # NaN yoksa True olur.
    # -----------------------------------------------------

    interval_frame = pd.concat(
        [
            forecast.rename("Forecast"),
            lower.rename("Lower"),
            upper.rename("Upper"),
        ],
        axis=1,
    )

    no_nan_check = not (
        interval_frame
        .isna()
        .any()
        .any()
    )


    # -----------------------------------------------------
    # 4. Google Trends 0–100 domain kontrolü
    # -----------------------------------------------------
    #
    # .ge(0):
    # greater than or equal
    # bütün değerler >= 0 mı?
    #
    # .le(100):
    # less than or equal
    # bütün değerler <= 100 mü?
    # -----------------------------------------------------

    lower_domain_check = (
        interval_frame
        .ge(0)
        .all()
        .all()
    )

    upper_domain_check = (
        interval_frame
        .le(100)
        .all()
        .all()
    )


    # -----------------------------------------------------
    # O trendin bütün kontrolleri
    # -----------------------------------------------------

    trend_pass = all(
        [
            lower_check,
            upper_check,
            no_nan_check,
            lower_domain_check,
            upper_domain_check,
        ]
    )


    interval_validation_results[
        forecast_column
    ] = trend_pass


    print(
        f"\n{forecast_column}"
    )

    print(
        "Lower <= Forecast:",
        lower_check,
    )

    print(
        "Forecast <= Upper:",
        upper_check,
    )

    print(
        "No NaN:",
        no_nan_check,
    )

    print(
        "All values >= 0:",
        lower_domain_check,
    )

    print(
        "All values <= 100:",
        upper_domain_check,
    )


ChatGPT
Lower <= Forecast: True
Forecast <= Upper: True
No NaN: True
All values >= 0: True
All values <= 100: True

Gemini
Lower <= Forecast: True
Forecast <= Upper: True
No NaN: True
All values >= 0: True
All values <= 100: True

Claude
Lower <= Forecast: True
Forecast <= Upper: True
No NaN: True
All values >= 0: True
All values <= 100: True


In [25]:
# ---------------------------------------------------------
# Genel interval validation sonucu
# ---------------------------------------------------------

all_intervals_valid = all(
    interval_validation_results.values()
)


if all_intervals_valid:

    print(
        "\n✅ PREDICTION INTERVAL / DOMAIN CHECK: PASS"
    )

else:

    print(
        "\n❌ PREDICTION INTERVAL / DOMAIN CHECK: FAIL"
    )


✅ PREDICTION INTERVAL / DOMAIN CHECK: PASS


## 6. Monitoring Sisteminin Doğrulanması

Final dashboard yalnızca gelecek tahminlerini değil, geçmişteki sıra dışı
hareketleri ve gelecekteki yükselen trend sinyalini de göstermektedir.

Bu aşamada:

- anomaly detection fonksiyonunun bütün trendlerde çalışması,
- son gözlem için geçerli bir anomaly sonucu üretmesi,
- rising trend signal fonksiyonunun gerekli çıktıları oluşturması,
- monitoring bileşenlerinde eksik veya geçersiz sonuç bulunmaması

kontrol edilecektir.

Buradaki amaç belirli bir trendin mutlaka anomaly veya yükselen trend
üretmesi değildir. Amaç monitoring sisteminin beklenen yapıda ve hatasız
çıktı üretebilmesidir.

In [26]:
# ---------------------------------------------------------
# 6. MONITORING VALIDATION
# ---------------------------------------------------------
#
# Dashboard'da iki monitoring fonksiyonumuz var:
#
# 1. detect_anomalies()
#    → geçmişte / güncel haftada sıra dışı hareket var mı?
#
# 2. detect_rising_trend_signal()
#    → gelecek 4 haftalık forecast belirgin biçimde
#      yükseliyor mu?
#
# Burada "mutlaka alarm çıkmalı" demiyoruz.
#
# Örneğin:
# is_rising_signal = False
#
# tamamen geçerli bir sonuçtur.
#
# Test ettiğimiz şey:
#
# "Fonksiyonlar doğru yapıda sonuç üretiyor mu?"
# ---------------------------------------------------------

from src.monitoring import (
    detect_anomalies,
    detect_rising_trend_signal,
)


# CSV'deki sütun adı ile forecast'taki
# kullanıcı dostu isimleri eşleştiriyoruz.
monitoring_trend_map = {
    "chatgpt": "ChatGPT",
    "gemini": "Gemini",
    "claude": "Claude",
}


# Her trendin PASS / FAIL sonucunu burada tutacağız.
monitoring_validation_results = {}


for data_column, forecast_column in monitoring_trend_map.items():

    # -----------------------------------------------------
    # Geçmiş gerçek seri
    # -----------------------------------------------------

    historical_series = updated_data[
        data_column
    ]


    # -----------------------------------------------------
    # Anomaly detection
    # -----------------------------------------------------
    #
    # Dashboard'da kullandığımız aynı parametreleri
    # burada da kullanıyoruz.
    # -----------------------------------------------------

    anomaly_result = detect_anomalies(
        series=historical_series,
        window=12,
        threshold=3.5,
        min_absolute_change=5.0,
    )


    # Son haftanın monitoring sonucunu alıyoruz.
    latest_anomaly = anomaly_result.iloc[-1]


    # -----------------------------------------------------
    # Anomaly çıktısı geçerli mi?
    # -----------------------------------------------------

    anomaly_length_check = (
        len(anomaly_result)
        == len(historical_series)
    )


    # Dashboard'un kullandığı temel kolonlar
    # gerçekten var mı?
    required_anomaly_columns = {
        "Value",
        "Anomaly_Score",
        "Is_Anomaly",
    }

    anomaly_columns_check = (
        required_anomaly_columns
        .issubset(anomaly_result.columns)
    )


    # Son haftanın anomaly score'u NaN olmamalı.
    latest_score_check = pd.notna(
        latest_anomaly["Anomaly_Score"]
    )


    # -----------------------------------------------------
    # Rising trend signal
    # -----------------------------------------------------

    forecast_series = pipeline_forecast[
        forecast_column
    ]

    current_value = float(
        historical_series.iloc[-1]
    )


    rising_signal = detect_rising_trend_signal(
        current_value=current_value,
        forecast=forecast_series,
        min_total_increase=5.0,
        min_positive_ratio=0.75,
    )


    # Dashboard şu iki anahtarı kullanıyor:
    #
    # is_rising_signal
    # total_increase
    #
    # Bu anahtarlar dictionary içinde gerçekten var mı?
    required_signal_keys = {
        "is_rising_signal",
        "total_increase",
    }

    signal_keys_check = (
        required_signal_keys
        .issubset(rising_signal.keys())
    )


    # total_increase sayısal bir değer olmalı.
    total_increase_check = pd.notna(
        rising_signal["total_increase"]
    )


    # -----------------------------------------------------
    # O trend için genel monitoring sonucu
    # -----------------------------------------------------

    trend_monitoring_pass = all(
        [
            anomaly_length_check,
            anomaly_columns_check,
            latest_score_check,
            signal_keys_check,
            total_increase_check,
        ]
    )


    monitoring_validation_results[
        forecast_column
    ] = trend_monitoring_pass


    # -----------------------------------------------------
    # Sonuçları görelim
    # -----------------------------------------------------

    print(
        f"\n--- {forecast_column} ---"
    )

    print(
        "Latest anomaly score:",
        round(
            float(latest_anomaly["Anomaly_Score"]),
            3,
        ),
    )

    print(
        "Is anomaly:",
        bool(latest_anomaly["Is_Anomaly"]),
    )

    print(
        "Forecast total increase:",
        round(
            float(rising_signal["total_increase"]),
            3,
        ),
    )

    print(
        "Rising signal:",
        rising_signal["is_rising_signal"],
    )

    print(
        "Monitoring structure valid:",
        trend_monitoring_pass,
    )


--- ChatGPT ---
Latest anomaly score: 0.472
Is anomaly: False
Forecast total increase: 2.023
Rising signal: False
Monitoring structure valid: True

--- Gemini ---
Latest anomaly score: -0.542
Is anomaly: False
Forecast total increase: 0.0
Rising signal: False
Monitoring structure valid: True

--- Claude ---
Latest anomaly score: -1.264
Is anomaly: False
Forecast total increase: 0.0
Rising signal: False
Monitoring structure valid: True


In [27]:
# ---------------------------------------------------------
# Üç trendin monitoring sonucu
# ---------------------------------------------------------

all_monitoring_valid = all(
    monitoring_validation_results.values()
)


if all_monitoring_valid:

    print(
        "\n✅ MONITORING CHECK: PASS"
    )

else:

    print(
        "\n❌ MONITORING CHECK: FAIL"
    )


✅ MONITORING CHECK: PASS


## 7. Final End-to-End Sistem Kontrolü

Önceki adımlarda sistemin farklı bileşenleri ayrı ayrı doğrulandı.

Bu son aşamada:

- gerekli dosyaların varlığı,
- veri ve metadata cutoff tarihinin uyumu,
- kaydedilmiş modellerin yüklenebilmesi,
- inference pipeline'ın aynı forecast'u yeniden üretebilmesi,
- prediction interval ve 0–100 domain kurallarının geçerliliği,
- monitoring sisteminin doğru yapıda çıktı üretmesi

tek bir genel sistem kontrolünde birleştirilmektedir.

Bütün kontrollerin başarılı olması, final forecasting sisteminin mevcut
model ve veri snapshot'ı üzerinde uçtan uca tutarlı çalıştığını gösterir.

In [28]:
# ---------------------------------------------------------
# 7. FINAL END-TO-END SYSTEM CHECK
# ---------------------------------------------------------
#
# Şimdi gün boyunca oluşturduğumuz bütün
# PASS / FAIL değişkenlerini tek yerde topluyoruz.
#
# Böylece sistemin genel durumunu:
#
# "Bütün ana bileşenler çalışıyor mu?"
#
# sorusuyla değerlendirebiliriz.
# ---------------------------------------------------------


# Her kontrolün sonucunu dictionary içinde tutuyoruz.
#
# Bu yapı daha sonra hangi kontrolün başarısız
# olduğunu tek tek görebilmemizi de sağlar.
final_system_checks = {
    "Required Files": all_required_files_exist,
    "Data / Metadata Cutoff": cutoff_matches,
    "Saved Model Loading": models_loaded_correctly,
    "Pipeline Reproducibility": pipeline_reproducible,
    "Prediction Interval / Domain": all_intervals_valid,
    "Monitoring": all_monitoring_valid,
}


# ---------------------------------------------------------
# Kontrol sonuçlarını ekranda gösterme
# ---------------------------------------------------------

for check_name, check_result in final_system_checks.items():

    # Eğer sonuç True ise PASS,
    # False ise FAIL yazıyoruz.
    status = (
        "✅ PASS"
        if check_result
        else "❌ FAIL"
    )

    print(
        f"{check_name}: {status}"
    )


# ---------------------------------------------------------
# Genel sistem sonucu
# ---------------------------------------------------------
#
# all():
# Dictionary'deki bütün Boolean değerlerin
# True olup olmadığını kontrol eder.
# ---------------------------------------------------------

end_to_end_system_valid = all(
    final_system_checks.values()
)


print("\n" + "-" * 50)


if end_to_end_system_valid:

    print(
        "✅ END-TO-END SYSTEM VALIDATION: PASS"
    )

else:

    print(
        "❌ END-TO-END SYSTEM VALIDATION: FAIL"
    )

Required Files: ✅ PASS
Data / Metadata Cutoff: ✅ PASS
Saved Model Loading: ✅ PASS
Pipeline Reproducibility: ✅ PASS
Prediction Interval / Domain: ✅ PASS
Monitoring: ✅ PASS

--------------------------------------------------
✅ END-TO-END SYSTEM VALIDATION: PASS


## Sonuç

End-to-end validation sonucunda bütün ana sistem kontrolleri başarıyla
tamamlandı.

Final sistem;

- gerekli artifact dosyalarını erişilebilir durumda tutmakta,
- veri ve model cutoff tarihlerini tutarlı kullanmakta,
- kaydedilmiş Prophet ve XGBoost modellerini yeniden yükleyebilmekte,
- kayıtlı final forecast'u aynı şekilde yeniden üretebilmekte,
- prediction interval değerlerini geçerli ve 0–100 sınırları içerisinde
  oluşturabilmekte,
- anomaly ve rising-trend monitoring bileşenlerini hatasız
  çalıştırabilmektedir.

Bu nedenle mevcut `2026-08-09` veri ve model snapshot'ı için
end-to-end sistem doğrulaması başarılı kabul edilmiştir.